# Atividade 2: Limpeza e Padronização de Dados

**Dataset escolhido:** [Google Playstore Apps](https://www.kaggle.com/datasets/lava18/google-play-store-apps)


**Grupo**

| Nome | RA |
|-|-|
|Carlos Eduardo Hayashi|10419790|
|Edson Luiz Cardoso Ohira|10419663|
|Thomaz de Souza Scopel|10417183|
|Vitor Tibães Santos|10418976|

Este dataset contém informações extraídas (web scraping) de mais de 10.000 aplicativos listados na loja oficial do Android. A base original apresenta 13 variáveis, incluindo o Gênero do aplicativo, Avaliação (Rating), Número de Instalações, Tamanho (Size) e Preço. 

Por ser um dado bruto gerado por raspagem de dados da web, ele simula perfeitamente um ambiente real de negócios: apresenta diversos ruídos de formatação, unidades de medida misturadas na mesma coluna, caracteres especiais junto a números e valores ausentes. O objetivo desta atividade é diagnosticar essas anomalias e aplicar as transformações necessárias para que a base possa ser consumida por modelos analíticos.

**Objetivo:** Aplicar técnicas de diagnóstico, limpeza e preparação de dados, justificando analiticamente cada etapa para garantir a qualidade de futuras modelagens.

## 0. Preparando Ambiente

In [9]:
!pip install pandas matplotlib seaborn kagglehub[pandas-datasets]


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
# pip install kagglehub[pandas-datasets]
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")

# Nome exato do arquivo principal dentro do dataset
file_path = "googleplaystore.csv"

# Carrega o dataset direto para um DataFrame do Pandas
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "lava18/google-play-store-apps",
  file_path,
)

# Inspecionando o carregamento
tamanho_original = df.shape
print(f"Dimensões originais: {tamanho_original}")
display(df.head())

/tmp/ipykernel_2985/1441028953.py:16: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Dimensões originais: (10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## 1. Diagnóstico de Qualidade Obrigatório
Antes de modificar os dados, inspecionamos a base para entender sua estrutura, identificar tipos incorretos inferidos pelo Pandas e mapear as inconsistências geradas no processo de extração.

In [11]:
display(df.info())
print("\nValores Ausentes:")
display(df.isnull().sum())
print(f"\nDuplicatas exatas: {df.duplicated().sum()}")

# Inspecionando o principal problema: Colunas numéricas lidas como texto (object)
display(df[['Installs', 'Price', 'Size']].head())

# Identificando o famoso 'outlier' de erro de raspagem neste dataset
display(df[df['Rating'] > 5])

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


None


Valores Ausentes:


App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64


Duplicatas exatas: 483


,Installs,Price,Size
0,"10,000+",0,19M
1,"500,000+",0,14M
2,"5,000,000+",0,8.7M
3,"50,000,000+",0,25M
4,"100,000+",0,2.8M


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


### Relatório de Diagnóstico Inicial
*   **Outlier / Erro Estrutural Crítico:** A inspeção (`df[df['Rating'] > 5]`) revelou uma anomalia na linha 10472 (App *Life Made WI-Fi Touchscreen Photo Frame*). Ocorreu um erro de raspagem (shift) onde os dados foram deslocados uma coluna para a direita. A nota (Rating) registrada foi 19.0 (o máximo é 5), o tamanho ficou como "1,000+" e o preço como "Everyone".
*   **Tipagem Incorreta:** Devido a esse deslocamento e à formatação de strings, colunas puramente numéricas como `Reviews`, `Installs` (que possuem `+` e `,`) e `Price` (que possui `$`) foram lidas pelo Pandas como texto (`object`).
*   **Ausências e Duplicatas:** Foram encontradas 483 linhas exatamente duplicadas. Além disso, a coluna `Rating` possui 1474 valores ausentes (NaN), indicando aplicativos sem volume suficiente de avaliações.

## 2. Limpeza e Preparação Obrigatório
Execução das transformações com base no diagnóstico estrutural.

In [12]:
# 1. Removendo a linha corrompida (Erro de Web Scraping)
df = df[df['Rating'] != 19.0].copy()
print(f"Shape após remover outlier/erro (1 linha): {df.shape}")

# 2. Remoção de Duplicatas Exatas
df = df.drop_duplicates()
print(f"Shape após remover duplicatas exatas: {df.shape}")

# 3. Limpeza de strings e conversão para numérico (Installs, Price e Reviews)
df['Installs'] = df['Installs'].str.replace('+', '').str.replace(',', '').astype(float)
df['Price'] = df['Price'].str.replace('$', '').astype(float)
df['Reviews'] = df['Reviews'].astype(int)

# 4. Uso do pd.to_datetime() conforme requisito da lição
df['Last Updated'] = pd.to_datetime(df['Last Updated'])
print(f"Shape após conversões de tipo (sem alteração de linhas): {df.shape}")

# 5. Tratamento de Nulos na coluna Rating (Imputação pela Mediana)
# A imputação preenche buracos, então o shape não muda, mas a contagem de não-nulos sim
mediana_rating = df['Rating'].median()
df['Rating'] = df['Rating'].fillna(mediana_rating)
print(f"Shape final após imputações: {df.shape}")

Shape após remover outlier/erro (1 linha): (10840, 13)
Shape após remover duplicatas exatas: (10357, 13)
Shape após conversões de tipo (sem alteração de linhas): (10357, 13)
Shape final após imputações: (10357, 13)


## 3. Justificativas das Decisões

*   **Outliers e Erros de Entrada:** A linha contendo a nota 19.0 foi removida. O valor era claramente um erro de entrada/raspagem de dados (já que a nota máxima é 5.0), causando o deslocamento de todas as variáveis daquela linha. A manutenção dessa linha corromperia a tipagem das colunas subsequentes.
*   **Duplicatas:** Foi aplicada a remoção direta (`drop_duplicates()`). Em uma base de aplicativos, múltiplas capturas do mesmo app enviesam as médias de preço e inflacionam falsamente a contagem de popularidade de certos gêneros.
*   **Inconsistências e Padronização:** O tratamento usando `str.replace()` foi necessário nas colunas de instalação e preço para remover formatações humanas (`+`, `,`, `$`), permitindo a conversão para numérico. Além disso, utilizamos `pd.to_datetime()` na coluna `Last Updated` para normalizar o dado de tempo, viabilizando futuras análises de séries temporais (como entender se apps atualizados recentemente têm notas maiores).
*   **Valores Ausentes:** Na coluna `Rating`, aplicamos a imputação pela mediana (`fillna`). A justificativa é que a distribuição de notas em lojas de aplicativos é fortemente assimétrica (concentrada em notas altas). A exclusão sumária (dropna) das linhas sem nota resultaria na perda excessiva de dados válidos nas outras colunas.

### Respostas aos Questionamentos Finais

**1. Há valores sentinela? Como você verificou?**
Sim. A nota `19.0` atuou na prática como um valor sentinela não-intencional. Verificamos isso inicialmente usando um filtro lógico (`df[df['Rating'] > 5]`) durante o diagnóstico, identificando a quebra do padrão da regra de negócios da loja (onde o limite real é 5.0).

**2. As ausências têm padrão?**
Sim. As ausências na coluna `Rating` geralmente ocorrem em aplicativos com baixíssimo volume de `Reviews` ou de `Installs`. Isso indica um padrão estrutural (MAR - Missing At Random): o app é tão novo ou obscuro que ainda não acumulou avaliações para gerar uma média.

**3. Quantos registros removeu, e qual o percentual?**
O dataset original possuía 10.841 registros. O dataset final possui 10.357. Foram removidos **484 registros** (sendo 1 erro de entrada e 483 duplicatas). Isso representa uma perda muito segura de aproximadamente **4,46%** da base de dados.

**4. Alguma variável só é conhecida após o desfecho?**
Sim. Se o "desfecho" (objetivo) de um futuro modelo de Machine Learning for prever o sucesso de um aplicativo *antes* de ele ser lançado na loja, as colunas `Rating`, `Reviews` e `Installs` representam **Vazamento de Dados (Data Leakage)**, pois essas métricas só passam a existir e ser conhecidas semanas ou meses *após* o lançamento público.